# Notebook 06 — SkinX offline model and inference checks

**Goal:** Convert the locked EfficientNetB0 model from Notebook 04 into a smaller TFLite model for offline screening, without changing the model or its validation-selected threshold. The mobile screening step should work without internet, FastAPI or a cloud account.

The notebook checks the locked files and preprocessing, tries several TFLite formats, compares them on the original conventional validation split, and selects the smallest candidate that does not reduce validation true positives or true negatives. It also checks the inference and input-handling code, exports the mobile files and optional Python/FastAPI reference, and records a descriptive PWA check.


In [2]:
from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, time, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, roc_auc_score
from IPython.display import display

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/SkinX')
SPLITS = ROOT / 'data/splits'
LOCKED_RUN = ROOT / 'models/runs/notebook_04/three_models_finetune_v1'
PWA_ROOT = ROOT / 'data/raw_data/PWA_clinical_images'
RUN_TAG = 'offline_tflite_cam_v2'   # New run after the CAM compatibility fix; earlier exports are kept.
OUT = ROOT / 'models/runs/notebook_06' / RUN_TAG

TRAIN_CSV = SPLITS / 'skinX_train_manifest_v1.csv'
VAL_CSV = SPLITS / 'skinX_val_manifest_v1.csv'
LOCK_JSON = LOCKED_RUN / 'locked_selection.json'
IMAGE_SIZE = (224, 224)
SEED = 42
CALIBRATION_PER_CLASS = 75        # Use training images only for calibration, not validation, test or PWA images.
UNCERTAINTY_MARGIN = 0.05         # Initial uncertainty margin, checked on conventional validation scores only.
NUM_THREADS = 2                   # Use a small thread count for Colab; test device performance separately.

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Destination:', OUT)


Mounted at /content/drive
TensorFlow: 2.21.0
Destination: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2


## 2. Check the locked model and data from Notebook 04

The conventional test split remains sealed. Training images are used for INT8 calibration, and validation images are used to compare deployment formats. PWA images are checked only after the deployment choice has been fixed.

In [3]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

for path in [TRAIN_CSV, VAL_CSV, LOCK_JSON]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required file: {path}')
lock = json.loads(LOCK_JSON.read_text(encoding='utf-8'))
MODEL_PATH = LOCKED_RUN / lock['model_file']
if not MODEL_PATH.is_file() or sha256(MODEL_PATH) != lock['model_sha256']:
    raise ValueError('Locked model missing or changed; do not continue.')
if lock['architecture'] != 'EfficientNetB0':
    raise ValueError('Expected locked EfficientNetB0 from Notebook 04')
if sha256(TRAIN_CSV) != lock['train_sha256'] or sha256(VAL_CSV) != lock['validation_sha256']:
    raise ValueError('Training/validation CSV changed since Notebook 04')
THRESHOLD = float(lock['threshold'])
assert 0 < THRESHOLD < 1

train = pd.read_csv(TRAIN_CSV)
val = pd.read_csv(VAL_CSV)
required = ['image_id', 'filepath', 'binary_label', 'label_group', 'group_id', 'md5_hash']
for name, frame in [('train', train), ('validation', val)]:
    if not set(required).issubset(frame.columns):
        raise ValueError(f'{name}: required CSV columns are missing')
    if frame[required].isna().any().any():
        raise ValueError(f'{name}: missing required fields')
    if not frame['binary_label'].isin([0, 1]).all():
        raise ValueError(f'{name}: incorrect binary labels')
    if not frame['binary_label'].eq(frame['label_group'].map({'NON_TARGET':0, 'TARGET':1})).all():
        raise ValueError(f'{name}: label mismatch')
    if frame['md5_hash'].duplicated().any() or frame['image_id'].duplicated().any():
        raise ValueError(f'{name}: duplicate IDs or hashes')
    absent = [p for p in frame['filepath'] if not Path(p).is_file()]
    if absent:
        raise FileNotFoundError(f'{name}: {len(absent)} missing files, example: {absent[0]}')
for col in ['image_id', 'md5_hash', 'group_id']:
    if set(train[col]) & set(val[col]):
        raise ValueError(f'Data leak: train and validation overlap in {col}')

# Stop if this deployment run already exists.
if OUT.exists() and any(OUT.iterdir()):
    raise FileExistsError(f'{OUT} already contains files. Set a NEW RUN_TAG.')
OUT.mkdir(parents=True, exist_ok=True)

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
if tuple(model.input_shape[1:]) != (224,224,3) or model.output_shape[-1] != 1:
    raise ValueError(f'Unexpected locked input/output: {model.input_shape} -> {model.output_shape}')
print('Original model:', lock['stage'])
print('Threshold:', THRESHOLD)
print('Training:', len(train), 'Validation:', len(val))
print('Output shape:', model.output_shape)


Original model: EfficientNetB0__head
Threshold: 0.374953031539917
Training: 1661 Validation: 339
Output shape: (None, 1)


## 3. Keep the training preprocessing

Training decoded images as RGB, kept float32 pixel values in the **0–255** range, and used `tf.image.resize_with_pad` to resize them to **224 × 224** with centered black padding. EfficientNetB0 preprocessing is already part of the locked model, so the pixels must not be divided by 255.

The Android app must create the same input tensor. Phone and gallery images also need their EXIF orientation corrected before resizing. A synthetic fixture is saved to check preprocessing and TFLite parity.

In [4]:
def load_image(path):
    data = tf.io.read_file(str(path))
    image = tf.io.decode_image(data, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(tf.cast(image, tf.float32), *IMAGE_SIZE)
    return image.numpy()

def get_images(frame):
    # Load one image at a time to limit memory use.
    for p in frame['filepath'].astype(str):
        yield load_image(p)

def metrics(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    y_pred = (scores >= THRESHOLD).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel())
    return {'n': int(len(y_true)), 'tp': tp, 'fn': fn, 'tn': tn, 'fp': fp,
            'sensitivity': tp/(tp+fn) if tp+fn else None,
            'specificity': tn/(tn+fp) if tn+fp else None,
            'auroc': float(roc_auc_score(y_true, scores)) if len(np.unique(y_true))==2 else None}

def keras_scores(frame):
    results=[]
    for img in get_images(frame):
        score=float(np.asarray(model(np.expand_dims(img,0), training=False)).reshape(-1)[0])
        results.append(score)
    return np.asarray(results, dtype=np.float32)

truth = val['binary_label'].astype(int).to_numpy()
keras_val = keras_scores(val)
reference = metrics(truth, keras_val)
expected = lock['validation_metrics']
if (reference['tp'], reference['tn']) != (expected['true_positives'], expected['true_negatives']):
    raise RuntimeError('Loaded Keras model does not reproduce Notebook 04 validation labels at locked threshold. Check environment before converting.')
print('Verified original Keras validation:')
display(pd.DataFrame([reference]))

# Use class-balanced calibration from training images only. Exclude PWA and test images.
calibration = (train.groupby('binary_label', group_keys=False)
               .sample(n=CALIBRATION_PER_CLASS, random_state=SEED)
               .sample(frac=1, random_state=SEED))
if len(calibration) != 2*CALIBRATION_PER_CLASS:
    raise ValueError('Not enough calibration images per class')

def representative_dataset():
    for image in get_images(calibration):
        yield [np.expand_dims(image, 0).astype(np.float32)]

print('INT8 calibration:', len(calibration), 'TRAIN images only')


Verified original Keras validation:


,n,tp,fn,tn,fp,sensitivity,specificity,auroc
0,339,269,17,43,10,0.940559,0.811321,0.947552


INT8 calibration: 150 TRAIN images only


## 4. Convert the model to offline formats

The notebook tries four options: float32, float16 weights, dynamic-range INT8 weights and full INT8. Full INT8 uses integer inputs and outputs and is calibrated with training images only. Conversion failures are recorded rather than hidden by enabling `SELECT_TF_OPS` or Flex, which would add complexity to the Flutter app.

Model size and speed are measured. Float16 may reduce file size without making inference faster on an older CPU, and INT8 may change predictions near the threshold. The results are compared before a format is selected.

In [5]:
variants = ['float32', 'float16', 'dynamic_int8', 'full_int8']
models_dir = OUT / 'candidates'
models_dir.mkdir(exist_ok=True)
conversion = {}

for variant in variants:
    print('Converting:', variant)
    started = time.perf_counter()
    try:
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
        if variant == 'float16':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.target_spec.supported_types = [tf.float16]
        elif variant == 'dynamic_int8':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
        elif variant == 'full_int8':
            converter.optimizations = [tf.lite.Optimize.DEFAULT]
            converter.representative_dataset = representative_dataset
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            converter.inference_input_type = tf.uint8
            converter.inference_output_type = tf.uint8
        raw = converter.convert()
        path = models_dir / f'skinx_{variant}.tflite'
        path.write_bytes(raw)
        conversion[variant] = {'status': 'converted', 'path': str(path),
                               'size_mb': round(path.stat().st_size/1024**2, 3),
                               'seconds': round(time.perf_counter()-started, 1)}
        print('  OK:', conversion[variant]['size_mb'], 'MiB')
    except Exception as e:
        conversion[variant] = {'status': 'FAILED', 'error': str(e)[:1200]}
        print('  Failed:', type(e).__name__, str(e)[:250])
    gc.collect()

(OUT/'conversion_report.json').write_text(json.dumps(conversion, indent=2)+'\n')
display(pd.DataFrame.from_dict(conversion, orient='index'))
if not any(v['status']=='converted' for v in conversion.values()):
    raise RuntimeError('No TFLite model could be converted. Keep the Keras model unchanged.')


Converting: float32
Saved artifact at '/tmp/tmpnpkrpc7o'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140405086684944: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  140405086684752: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  140405114682064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085521104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085520336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085520912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085521296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085521872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085522256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085522064: TensorSpec(shape=(), dtype=tf.resour

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


  OK: 4.705 MiB


,status,path,size_mb,seconds
float32,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,15.320,15.6
float16,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,7.754,15.6
dynamic_int8,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,4.360,15.8
full_int8,converted,/content/drive/MyDrive/SkinX/models/runs/noteb...,4.705,172.4


## 5. Check the TFLite interpreter and input types

The app must check the input tensor type. Float models use RGB `float32` values from 0 to 255. Integer models need quantization using the input scale and zero point, and their outputs need to be dequantized. Do not assume the quantization settings or normalize the input to 0–1.

In [6]:
def interpreter_for(path):
    interpreter = tf.lite.Interpreter(model_path=str(path), num_threads=NUM_THREADS)
    interpreter.allocate_tensors()
    ins, outs = interpreter.get_input_details(), interpreter.get_output_details()
    if len(ins)!=1 or len(outs)!=1 or tuple(ins[0]['shape'])!=(1,224,224,3):
        raise ValueError('Unexpected TFLite input/output tensor contract')
    if int(np.prod(outs[0]['shape'])) != 1:
        raise ValueError('Expected one output probability')
    op_names = [op['op_name'] for op in interpreter._get_ops_details()]
    if any(op.startswith('Flex') or 'CUSTOM' in op for op in op_names):
        raise ValueError('TFLite uses Flex/custom ops; not suitable for default lightweight runtime')
    return interpreter, ins[0], outs[0]

def run_one(interpreter, inp, out, image):
    data = np.expand_dims(image,0).astype(np.float32)
    if np.issubdtype(inp['dtype'], np.integer):
        scale, zero = inp['quantization']
        if scale<=0: raise ValueError('Invalid quantized input scale')
        bounds = np.iinfo(inp['dtype'])
        data = np.clip(np.rint(data/scale+zero), bounds.min, bounds.max).astype(inp['dtype'])
    else:
        data = data.astype(inp['dtype'])
    interpreter.set_tensor(inp['index'], data)
    interpreter.invoke()
    p = interpreter.get_tensor(out['index'])
    if np.issubdtype(out['dtype'], np.integer):
        scale, zero = out['quantization']
        if scale<=0: raise ValueError('Invalid quantized output scale')
        p = (p.astype(np.float32)-zero)*scale
    score = float(np.asarray(p).reshape(-1)[0])
    if not np.isfinite(score) or score < -0.005 or score > 1.005:
        raise ValueError('Invalid output probability')
    return min(1.0, max(0.0, score))

# Run a quick check before evaluating the full validation set.
example_image = load_image(val.iloc[0]['filepath'])
ready = {}
for variant, info in conversion.items():
    if info['status'] != 'converted': continue
    try:
        intr, inp, out = interpreter_for(info['path'])
        score = run_one(intr, inp, out, example_image)
        ready[variant] = True
        info['input_dtype'] = str(inp['dtype'])
        info['output_dtype'] = str(out['dtype'])
        info['input_quantization'] = list(map(float, inp['quantization']))
        info['output_quantization'] = list(map(float, out['quantization']))
        print(f'{variant}: first score {score:.5f} | input {inp["dtype"]} | output {out["dtype"]}')
    except Exception as e:
        ready[variant] = False
        info['status'] = 'FAILED_SMOKE_TEST'
        info['error'] = str(e)[:1200]
        print(f'{variant}: not usable — {e}')


float32: first score 0.86364 | input <class 'numpy.float32'> | output <class 'numpy.float32'>


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


float16: first score 0.86393 | input <class 'numpy.float32'> | output <class 'numpy.float32'>
dynamic_int8: first score 0.90437 | input <class 'numpy.float32'> | output <class 'numpy.float32'>
full_int8: first score 0.91406 | input <class 'numpy.uint8'> | output <class 'numpy.uint8'>


## 6. Compare models at the original threshold

Each usable TFLite model is evaluated on the same conventional validation images, using the original threshold. The threshold is not adjusted to improve a converted model. A candidate is accepted only if it has no fewer true positives and no fewer true negatives than the verified Keras model. This checks the conversion; it does not guarantee clinical performance.

In [7]:
report=[]
val_predictions=val[['image_id','filepath','binary_label','diagnostic']].copy()
val_predictions['keras_probability'] = keras_val

for variant, good in ready.items():
    if not good: continue
    info=conversion[variant]
    intr, inp, out=interpreter_for(info['path'])
    # The first load and inference include setup time; this short timing is only a Colab estimate.
    for _ in range(3): run_one(intr, inp, out, example_image)
    times=[]
    for _ in range(20):
        t0=time.perf_counter()
        run_one(intr, inp, out, example_image)
        times.append((time.perf_counter()-t0)*1000)
    scores=np.asarray([run_one(intr,inp,out,img) for img in get_images(val)], dtype=np.float32)
    stats=metrics(truth, scores)
    accepted=stats['tp']>=reference['tp'] and stats['tn']>=reference['tn']
    info['accepted_on_validation']=bool(accepted)
    info['median_colab_inference_ms']=round(float(np.median(times)),2)
    info['max_absolute_probability_difference']=float(np.max(np.abs(scores-keras_val)))
    val_predictions[f'{variant}_probability']=scores
    report.append({'variant':variant,'size_mb':info['size_mb'],
                   'median_colab_inference_ms':info['median_colab_inference_ms'],
                   'max_probability_difference':info['max_absolute_probability_difference'],
                   'accepted':accepted,**stats})

compare=pd.DataFrame(report).sort_values('size_mb') if report else pd.DataFrame()
compare.to_csv(OUT/'validation_comparison.csv', index=False)
val_predictions.to_csv(OUT/'validation_probabilities.csv', index=False)
(OUT/'conversion_report.json').write_text(json.dumps(conversion,indent=2)+'\n')
print('Reference Keras confusion:', reference)
display(compare)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use 

Reference Keras confusion: {'n': 339, 'tp': 269, 'fn': 17, 'tn': 43, 'fp': 10, 'sensitivity': 0.9405594405594405, 'specificity': 0.8113207547169812, 'auroc': 0.9475524475524476}


,variant,size_mb,median_colab_inference_ms,max_probability_difference,accepted,n,tp,fn,tn,fp,sensitivity,specificity,auroc
2,dynamic_int8,4.360,16.51,0.119101,False,339,268,18,41,12,0.937063,0.773585,0.946497
3,full_int8,4.705,7.30,0.527795,False,339,267,19,33,20,0.933566,0.622642,0.922054
1,float16,7.754,10.86,0.004999,True,339,269,17,43,10,0.940559,0.811321,0.947289
0,float32,15.320,11.05,0.002024,False,339,268,18,43,10,0.937063,0.811321,0.947486


## 7. Select and freeze one deployment model

Choose the smallest TFLite candidate that passes the validation check. If sizes are tied, prefer the one with lower Colab latency. If every conversion reduces validation performance, do not accept a weaker model just to get a smaller file. Any further conversion work should use a new run and training/validation data only. The held-out PWA and conventional test images must not be used for tuning.

In [8]:
accepted = compare[compare['accepted']].copy() if len(compare) else pd.DataFrame()
if accepted.empty:
    print('STOP: No TFLite variant preserved both TP and TN on conventional validation.')
    print('Do not ship one of these models. Review conversion/compatibility in a new run.')
    DEPLOY_READY = False
else:
    chosen=accepted.sort_values(['size_mb','median_colab_inference_ms']).iloc[0]
    variant=str(chosen['variant'])
    source_path=Path(conversion[variant]['path'])
    deploy_dir=OUT/'deployment'
    deploy_dir.mkdir(exist_ok=False)
    dest=deploy_dir/'skinx_screening.tflite'
    shutil.copy2(source_path,dest)
    intr,inp,out=interpreter_for(dest)

    # Compare candidate uncertainty margins using conventional validation scores only.
    # The 0.05 margin is an initial app setting, not a clinically validated rule.
    uncertainty_scores = val_predictions[f'{variant}_probability'].to_numpy(dtype=np.float64)
    def evaluate_uncertainty_margin(margin):
        abstain = np.abs(uncertainty_scores - THRESHOLD) <= margin
        covered = ~abstain
        predicted = (uncertainty_scores >= THRESHOLD).astype(int)
        tp = int(((predicted == 1) & (truth == 1) & covered).sum())
        fn = int(((predicted == 0) & (truth == 1) & covered).sum())
        tn = int(((predicted == 0) & (truth == 0) & covered).sum())
        fp = int(((predicted == 1) & (truth == 0) & covered).sum())
        target_abstained = int((abstain & (truth == 1)).sum())
        non_target_abstained = int((abstain & (truth == 0)).sum())
        target_count = int((truth == 1).sum())
        non_target_count = int((truth == 0).sum())
        return {
            'margin': float(margin),
            'lower_boundary': float(max(0.0, THRESHOLD - margin)),
            'upper_boundary': float(min(1.0, THRESHOLD + margin)),
            'n': int(len(truth)),
            'target_count': target_count,
            'non_target_count': non_target_count,
            'abstained': int(abstain.sum()),
            'abstention_rate': float(abstain.mean()),
            'classified': int(covered.sum()),
            'coverage_rate': float(covered.mean()),
            'target_abstained': target_abstained,
            'non_target_abstained': non_target_abstained,
            'tp_classified': tp,
            'fn_classified': fn,
            'tn_classified': tn,
            'fp_classified': fp,
            'sensitivity_among_classified': float(tp/(tp+fn)) if tp+fn else None,
            'specificity_among_classified': float(tn/(tn+fp)) if tn+fp else None,
            'target_flagged_for_review_if_cannot_infer_prompts_assessment': int(tp+target_abstained),
            'target_review_rate_if_cannot_infer_prompts_assessment': float((tp+target_abstained)/target_count),
            'non_target_flagged_for_review_if_cannot_infer_prompts_assessment': int(fp+non_target_abstained),
            'non_target_review_rate_if_cannot_infer_prompts_assessment': float((fp+non_target_abstained)/non_target_count),
        }

    uncertainty_comparison = [evaluate_uncertainty_margin(m) for m in [0.01, 0.025, UNCERTAINTY_MARGIN, 0.075, 0.10]]
    uncertainty_frame = pd.DataFrame(uncertainty_comparison)
    uncertainty_frame.to_csv(OUT/'uncertainty_policy_comparison.csv', index=False)
    selected_uncertainty = evaluate_uncertainty_margin(UNCERTAINTY_MARGIN)
    uncertainty_report = {
        'status': 'EVALUATED_NOT_CLINICALLY_VALIDATED',
        'threshold': THRESHOLD,
        'threshold_unchanged': True,
        'selected_margin': UNCERTAINTY_MARGIN,
        'selected_policy': selected_uncertainty,
        'candidate_comparison': uncertainty_comparison,
        'note': 'The uncertainty margin is an initial prototype decision policy. Cannot infer prompts the user to seek professional assessment; this can increase follow-up recommendations for NON_TARGET cases.'
    }
    (OUT/'uncertainty_policy_validation.json').write_text(json.dumps(uncertainty_report, indent=2)+'\n', encoding='utf-8')
    display(uncertainty_frame)

    config={
        'schema_version': 1,
        'model_name': 'SkinX offline binary screening',
        'selected_variant': variant,
        'source_architecture': lock['architecture'],
        'source_stage': lock['stage'],
        'source_model_sha256': lock['model_sha256'],
        'tflite_sha256': sha256(dest),
        'model_size_mib': round(dest.stat().st_size/(1024**2), 3),
        'input_shape': [1,224,224,3],
        'input_color_order': 'RGB',
        'external_normalization': 'NONE; pixel values are float32 0..255 before tensor quantization',
        'resize': 'tf.image.resize_with_pad, bilinear, preserve aspect ratio, black padding, centered',
        'input_dtype': str(np.dtype(inp['dtype'])),
        'output_dtype': str(np.dtype(out['dtype'])),
        'input_quantization': {'scale':float(inp['quantization'][0]), 'zero_point':int(inp['quantization'][1])},
        'output_quantization': {'scale':float(out['quantization'][0]), 'zero_point':int(out['quantization'][1])},
        'positive_label': 1,
        'labels': {'0':'NON_TARGET','1':'TARGET'},
        'target_probability_threshold': THRESHOLD,
        'decision_rule': 'Cannot infer when abs(target score - target_probability_threshold) <= uncertainty_margin; otherwise Suspicious above the upper boundary and Less suspicious below the lower boundary.',
        'cannot_infer': 'Technical failures or a score within the configured uncertainty band. A borderline score does not mean the lesion is harmless.',
        'uncertainty_abstention_enabled': True,
        'uncertainty_margin': UNCERTAINTY_MARGIN,
        'uncertainty_lower_boundary': max(0.0, THRESHOLD - UNCERTAINTY_MARGIN),
        'uncertainty_upper_boundary': min(1.0, THRESHOLD + UNCERTAINTY_MARGIN),
        'uncertainty_policy_validation': selected_uncertainty,
        'uncertainty_policy_validation_scope': 'Conventional validation split only; exploratory prototype policy, not clinical validation.',
        'validation_metrics': {k: (bool(v) if isinstance(v,np.bool_) else float(v) if isinstance(v,np.floating) else int(v) if isinstance(v,np.integer) else v) for k,v in chosen.to_dict().items()},
        'tensorflow_version':tf.__version__,
        'limitations':'Research pilot; not diagnostic; PWA data sparse and source-reported; do not hide clinical warning for non-target predictions.',
        'connectivity':'Screening inference is entirely offline when this model and preprocessing are bundled in the Android app.'
    }
    (deploy_dir/'skinx_model_config.json').write_text(json.dumps(config,indent=2)+'\n')
    (deploy_dir/'model_sha256.txt').write_text(config['tflite_sha256']+'  skinx_screening.tflite\n')
    DEPLOY_READY=True
    print('Selected:',variant,'|',config['model_size_mib'],'MiB')
    print('Deployment files:',deploy_dir)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,margin,lower_boundary,upper_boundary,n,target_count,non_target_count,abstained,abstention_rate,classified,coverage_rate,...,tp_classified,fn_classified,tn_classified,fp_classified,sensitivity_among_classified,specificity_among_classified,target_flagged_for_review_if_cannot_infer_prompts_assessment,target_review_rate_if_cannot_infer_prompts_assessment,non_target_flagged_for_review_if_cannot_infer_prompts_assessment,non_target_review_rate_if_cannot_infer_prompts_assessment
0,0.010,0.364953,0.384953,339,286,53,2,0.005900,337,0.994100,...,268,17,43,9,0.940351,0.826923,269,0.940559,10,0.188679
1,0.025,0.349953,0.399953,339,286,53,4,0.011799,335,0.988201,...,267,17,42,9,0.940141,0.823529,269,0.940559,11,0.207547
2,0.050,0.324953,0.424953,339,286,53,9,0.026549,330,0.973451,...,266,17,40,7,0.939929,0.851064,269,0.940559,13,0.245283
3,0.075,0.299953,0.449953,339,286,53,16,0.047198,323,0.952802,...,263,15,38,7,0.946043,0.844444,271,0.947552,15,0.283019
4,0.100,0.274953,0.474953,339,286,53,21,0.061947,318,0.938053,...,261,14,36,7,0.949091,0.837209,272,0.951049,17,0.320755


Selected: float16 | 7.754 MiB
Deployment files: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/deployment


## 8. Check the CAM explanation and export it separately

This section does not train or fine-tune the classifier and does not use the sealed test set. It compares the EfficientNetB0 class activation map (CAM) with Grad-CAM on a small, fixed sample from the conventional validation split. It then exports a separate float16 TFLite model with the target score followed by the row-major CAM map. The original deployment model and threshold remain available as a fallback.

The heatmap shows which image regions influence the model; it is not a lesion boundary or proof that the highlighted tissue is cancerous. The export passes only if the Keras score matches the locked model, validation true positives and true negatives do not fall, and the TFLite map remains close to the Keras map.

In [9]:
CAM_EXPORT_READY = False
CAM_VALIDATION_REPORT = {'status': 'SKIPPED', 'reason': 'Base TFLite conversion was not accepted.'}

if DEPLOY_READY:
    try:
        backbone_candidates = [
            layer for layer in model.layers
            if isinstance(layer, tf.keras.Model) and 'efficientnet' in layer.name.lower()
        ]
        if len(backbone_candidates) != 1:
            raise ValueError('Could not identify exactly one EfficientNetB0 backbone.')
        backbone = backbone_candidates[0]
        if 'top_activation' not in [layer.name for layer in backbone.layers]:
            raise ValueError('EfficientNetB0 top_activation layer was not found.')
        top_activation = backbone.get_layer('top_activation')
        feature_extractor = tf.keras.Model(
            inputs=backbone.input,
            outputs=top_activation.output,
            name='skinx_cam_feature_extractor',
        )
        dense_head = model.get_layer('probability_target')
        if dense_head.units != 1 or dense_head.activation.__name__ != 'sigmoid':
            raise ValueError('Expected the locked single-output sigmoid classifier head.')

        # Build the CAM from the final convolutional feature maps; dropout is inactive during inference.
        cam_inputs = model.input
        feature_maps = feature_extractor(cam_inputs, training=False)
        pooled = tf.keras.layers.GlobalAveragePooling2D(name='cam_global_average_pooling')(feature_maps)
        cam_probability = dense_head(pooled)
        # Use Keras operations to build the symbolic Functional model in Keras 3.
        # Direct TensorFlow calls such as tf.reshape and tf.reduce_sum do not work on a KerasTensor here.
        channel_weights = tf.keras.ops.reshape(dense_head.kernel[:, 0], (1, 1, 1, -1))
        weighted_features = tf.keras.ops.multiply(feature_maps, channel_weights)
        raw_cam = tf.keras.ops.sum(weighted_features, axis=-1)
        positive_cam = tf.keras.ops.relu(raw_cam)
        flat_cam = tf.keras.layers.Flatten(name='cam_row_major_flatten')(positive_cam)
        combined_output = tf.keras.layers.Concatenate(name='target_score_then_cam')([
            cam_probability, flat_cam
        ])
        cam_model = tf.keras.Model(cam_inputs, combined_output, name='skinx_efficientnet_cam')

        cam_shape = tuple(int(v) for v in positive_cam.shape[1:])
        if len(cam_shape) != 2 or min(cam_shape) < 1:
            raise ValueError(f'Unexpected CAM shape: {cam_shape}')
        cam_height, cam_width = cam_shape
        expected_output_elements = 1 + cam_height * cam_width
        if int(cam_model.output_shape[-1]) != expected_output_elements:
            raise ValueError('The combined score/CAM output shape is inconsistent.')

        def normalized_heatmap(values):
            values = np.maximum(np.asarray(values, dtype=np.float32), 0.0)
            maximum = float(np.max(values)) if values.size else 0.0
            return values / maximum if maximum > 1e-12 else np.zeros_like(values)

        # Compare CAM with Grad-CAM on eight fixed validation images; do not use PWA or test images.
        gradcam_errors = []
        for image in get_images(val.head(min(8, len(val)))):
            batch = tf.convert_to_tensor(np.expand_dims(image, 0), dtype=tf.float32)
            with tf.GradientTape() as tape:
                maps = feature_extractor(batch, training=False)
                tape.watch(maps)
                pooled_features = tf.reduce_mean(maps, axis=[1, 2])
                score = dense_head(pooled_features, training=False)[:, 0]
            gradients = tape.gradient(score, maps)
            if gradients is None:
                raise RuntimeError('Grad-CAM gradients could not be computed.')
            grad_weights = tf.reduce_mean(gradients, axis=[1, 2], keepdims=True)
            gradcam_map = tf.nn.relu(tf.reduce_sum(maps * grad_weights, axis=-1))[0].numpy()
            cam_map = tf.nn.relu(tf.reduce_sum(
                maps * tf.reshape(dense_head.kernel[:, 0], [1, 1, 1, -1]), axis=-1
            ))[0].numpy()
            gradcam_errors.append(float(np.mean(np.abs(
                normalized_heatmap(gradcam_map) - normalized_heatmap(cam_map)
            ))))
        max_gradcam_mae = float(max(gradcam_errors)) if gradcam_errors else float('inf')
        if max_gradcam_mae > 1e-3:
            raise RuntimeError(f'CAM did not match normalized Grad-CAM closely enough (max MAE={max_gradcam_mae:.6f}).')

        # Check that the CAM model score matches the locked Keras model on the full validation set.
        rebuilt_scores = []
        for image in get_images(val):
            vector = np.asarray(cam_model(np.expand_dims(image, 0), training=False)).reshape(-1)
            rebuilt_scores.append(float(vector[0]))
        rebuilt_scores = np.asarray(rebuilt_scores, dtype=np.float32)
        score_max_difference = float(np.max(np.abs(rebuilt_scores - keras_val)))
        rebuilt_stats = metrics(truth, rebuilt_scores)
        if score_max_difference > 1e-5 or (rebuilt_stats['tp'], rebuilt_stats['tn']) != (reference['tp'], reference['tn']):
            raise RuntimeError('The CAM model score branch did not reproduce the locked Keras model.')

        # Save the CAM model separately so the original TFLite model stays unchanged.
        cam_converter = tf.lite.TFLiteConverter.from_keras_model(cam_model)
        cam_converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
        cam_converter.optimizations = [tf.lite.Optimize.DEFAULT]
        cam_converter.target_spec.supported_types = [tf.float16]
        cam_raw = cam_converter.convert()
        cam_dest = deploy_dir / 'skinx_screening_cam.tflite'
        cam_dest.write_bytes(cam_raw)

        cam_interpreter = tf.lite.Interpreter(model_path=str(cam_dest), num_threads=NUM_THREADS)
        cam_interpreter.allocate_tensors()
        cam_inputs = cam_interpreter.get_input_details()
        cam_outputs = cam_interpreter.get_output_details()
        if len(cam_inputs) != 1 or len(cam_outputs) != 1:
            raise ValueError('CAM TFLite model must have one input and one combined output.')
        if tuple(cam_inputs[0]['shape']) != (1, 224, 224, 3):
            raise ValueError(f'Unexpected CAM TFLite input shape: {cam_inputs[0]["shape"]}')
        if int(np.prod(cam_outputs[0]['shape'])) != expected_output_elements:
            raise ValueError(f'Unexpected CAM TFLite output shape: {cam_outputs[0]["shape"]}')
        cam_ops = [op['op_name'] for op in cam_interpreter._get_ops_details()]
        if any(op.startswith('Flex') or 'CUSTOM' in op for op in cam_ops):
            raise ValueError('CAM TFLite uses Flex/custom ops, which are not supported by the current Flutter runtime.')

        def run_cam_output(image):
            tensor = np.expand_dims(image, 0).astype(np.float32)
            inp = cam_inputs[0]
            if np.issubdtype(inp['dtype'], np.integer):
                scale, zero = inp['quantization']
                if scale <= 0:
                    raise ValueError('Invalid CAM model input quantization.')
                limits = np.iinfo(inp['dtype'])
                tensor = np.clip(np.rint(tensor / scale + zero), limits.min, limits.max).astype(inp['dtype'])
            else:
                tensor = tensor.astype(inp['dtype'])
            cam_interpreter.set_tensor(inp['index'], tensor)
            cam_interpreter.invoke()
            vector = cam_interpreter.get_tensor(cam_outputs[0]['index'])
            if np.issubdtype(cam_outputs[0]['dtype'], np.integer):
                scale, zero = cam_outputs[0]['quantization']
                if scale <= 0:
                    raise ValueError('Invalid CAM model output quantization.')
                vector = (vector.astype(np.float32) - zero) * scale
            vector = np.asarray(vector, dtype=np.float32).reshape(-1)
            if len(vector) != expected_output_elements or not np.isfinite(vector).all():
                raise ValueError('Invalid CAM model output vector.')
            if vector[0] < -0.005 or vector[0] > 1.005:
                raise ValueError('CAM model returned an invalid target score.')
            vector[0] = np.clip(vector[0], 0.0, 1.0)
            return vector

        cam_tflite_scores = []
        tflite_map_mae = []
        tflite_map_cosine = []
        for index, image in enumerate(get_images(val)):
            vector = run_cam_output(image)
            cam_tflite_scores.append(float(vector[0]))
            if index < min(8, len(val)):
                keras_vector = np.asarray(cam_model(np.expand_dims(image, 0), training=False)).reshape(-1)
                keras_map = normalized_heatmap(keras_vector[1:].reshape(cam_height, cam_width))
                tflite_map = normalized_heatmap(vector[1:].reshape(cam_height, cam_width))
                tflite_map_mae.append(float(np.mean(np.abs(keras_map - tflite_map))))
                denom = float(np.linalg.norm(keras_map) * np.linalg.norm(tflite_map))
                cosine = float(np.sum(keras_map * tflite_map) / denom) if denom > 1e-12 else 1.0
                tflite_map_cosine.append(cosine)
        cam_tflite_scores = np.asarray(cam_tflite_scores, dtype=np.float32)
        cam_tflite_stats = metrics(truth, cam_tflite_scores)
        cam_tflite_max_score_difference = float(np.max(np.abs(cam_tflite_scores - keras_val)))
        max_tflite_map_mae = float(max(tflite_map_mae)) if tflite_map_mae else float('inf')
        min_tflite_map_cosine = float(min(tflite_map_cosine)) if tflite_map_cosine else -1.0

        def evaluate_cam_uncertainty_margin(margin):
            abstain = np.abs(cam_tflite_scores - THRESHOLD) <= margin
            covered = ~abstain
            predicted = (cam_tflite_scores >= THRESHOLD).astype(int)
            tp = int(((predicted == 1) & (truth == 1) & covered).sum())
            fn = int(((predicted == 0) & (truth == 1) & covered).sum())
            tn = int(((predicted == 0) & (truth == 0) & covered).sum())
            fp = int(((predicted == 1) & (truth == 0) & covered).sum())
            target_abstained = int((abstain & (truth == 1)).sum())
            non_target_abstained = int((abstain & (truth == 0)).sum())
            target_count = int((truth == 1).sum())
            non_target_count = int((truth == 0).sum())
            return {
                'margin': float(margin),
                'lower_boundary': float(max(0.0, THRESHOLD - margin)),
                'upper_boundary': float(min(1.0, THRESHOLD + margin)),
                'n': int(len(truth)),
                'target_count': target_count,
                'non_target_count': non_target_count,
                'abstained': int(abstain.sum()),
                'abstention_rate': float(abstain.mean()),
                'classified': int(covered.sum()),
                'coverage_rate': float(covered.mean()),
                'target_abstained': target_abstained,
                'non_target_abstained': non_target_abstained,
                'tp_classified': tp,
                'fn_classified': fn,
                'tn_classified': tn,
                'fp_classified': fp,
                'sensitivity_among_classified': float(tp/(tp+fn)) if tp+fn else None,
                'specificity_among_classified': float(tn/(tn+fp)) if tn+fp else None,
                'target_flagged_for_review_if_cannot_infer_prompts_assessment': int(tp+target_abstained),
                'target_review_rate_if_cannot_infer_prompts_assessment': float((tp+target_abstained)/target_count),
                'non_target_flagged_for_review_if_cannot_infer_prompts_assessment': int(fp+non_target_abstained),
                'non_target_review_rate_if_cannot_infer_prompts_assessment': float((fp+non_target_abstained)/non_target_count),
            }

        cam_uncertainty_comparison = [evaluate_cam_uncertainty_margin(m) for m in [0.01, 0.025, UNCERTAINTY_MARGIN, 0.075, 0.10]]
        cam_uncertainty_selected = evaluate_cam_uncertainty_margin(UNCERTAINTY_MARGIN)
        pd.DataFrame(cam_uncertainty_comparison).to_csv(OUT/'cam_uncertainty_policy_comparison.csv', index=False)

        base_tp = int(config['validation_metrics']['tp'])
        base_tn = int(config['validation_metrics']['tn'])
        if cam_tflite_max_score_difference > 0.02:
            raise RuntimeError(f'CAM TFLite score drift exceeded 0.02 (max difference={cam_tflite_max_score_difference:.5f}).')
        if cam_tflite_stats['tp'] < base_tp or cam_tflite_stats['tn'] < base_tn:
            raise RuntimeError(
                'CAM TFLite reduced validation true positives or true negatives compared with the selected base TFLite model; '
                'it will not be offered for the app.'
            )
        if max_tflite_map_mae > 0.08 or min_tflite_map_cosine < 0.98:
            raise RuntimeError(
                f'TFLite heatmap parity did not pass (max MAE={max_tflite_map_mae:.4f}, '
                f'min cosine={min_tflite_map_cosine:.4f}).'
            )

        cam_input_detail = cam_inputs[0]
        cam_output_detail = cam_outputs[0]
        cam_config = json.loads((deploy_dir / 'skinx_model_config.json').read_text(encoding='utf-8'))
        cam_config['schema_version'] = 2
        cam_config['model_name'] = 'SkinX offline binary screening with CAM explanation'
        cam_config['selected_variant'] = 'float16_cam'
        cam_config['tflite_sha256'] = sha256(cam_dest)
        cam_config['model_size_mib'] = round(cam_dest.stat().st_size / (1024 ** 2), 3)
        cam_config['output_dtype'] = str(np.dtype(cam_output_detail['dtype']))
        cam_config['output_shape'] = [1, expected_output_elements]
        cam_config['output_quantization'] = {
            'scale': float(cam_output_detail['quantization'][0]),
            'zero_point': int(cam_output_detail['quantization'][1]),
        }
        cam_config['cam'] = {
            'enabled': True,
            'height': cam_height,
            'width': cam_width,
            'output_layout': 'target_score_then_row_major_cam',
            'method': 'EfficientNetB0 class activation map (CAM)',
            'validated_against_gradcam': True,
            'gradcam_validation': {
                'sample_count': len(gradcam_errors),
                'max_normalized_map_mae': max_gradcam_mae,
            },
            'tflite_map_parity': {
                'sample_count': len(tflite_map_mae),
                'max_normalized_map_mae': max_tflite_map_mae,
                'minimum_cosine_similarity': min_tflite_map_cosine,
                'max_score_absolute_difference': cam_tflite_max_score_difference,
            },
            'validation_metrics': cam_tflite_stats,
            'selected_base_tflite_validation_metrics': config['validation_metrics'],
            'note': 'Highlighted areas influenced the model output; they are not a confirmed lesion boundary or diagnosis.',
        }
        cam_config_path = deploy_dir / 'skinx_cam_model_config.json'
        cam_config_path.write_text(json.dumps(cam_config, indent=2) + '\n', encoding='utf-8')
        CAM_EXPORT_READY = True
        CAM_VALIDATION_REPORT = {
            'status': 'PASSED',
            'keras_cam_gradcam_max_normalized_mae': max_gradcam_mae,
            'keras_score_max_absolute_difference': score_max_difference,
            'tflite_cam_max_normalized_mae': max_tflite_map_mae,
            'tflite_score_max_absolute_difference': cam_tflite_max_score_difference,
            'tflite_cam_minimum_cosine_similarity': min_tflite_map_cosine,
            'tflite_validation_metrics': cam_tflite_stats,
            'uncertainty_policy_validation': cam_uncertainty_selected,
            'model_path': str(cam_dest),
            'config_path': str(cam_config_path),
        }
        (OUT / 'cam_validation_report.json').write_text(json.dumps(CAM_VALIDATION_REPORT, indent=2) + '\n', encoding='utf-8')
        print('CAM export passed the Grad-CAM, score, TFLite map and validation checks.')
        print('CAM model:', cam_dest)
        display(pd.DataFrame([CAM_VALIDATION_REPORT]))
    except Exception as error:
        CAM_EXPORT_READY = False
        CAM_VALIDATION_REPORT = {'status': 'FAILED', 'error': f'{type(error).__name__}: {error}'}
        (OUT / 'cam_validation_report.json').write_text(json.dumps(CAM_VALIDATION_REPORT, indent=2) + '\n', encoding='utf-8')
        print('CAM export was not accepted. The original screening model remains unchanged.')
        print(CAM_VALIDATION_REPORT['error'])
else:
    print('CAM export skipped because no base TFLite model passed validation.')


Saved artifact at '/tmp/tmpxvlwy3tg'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')
Output Type:
  TensorSpec(shape=(None, 50), dtype=tf.float32, name=None)
Captures:
  140405086684944: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  140405086684752: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  140405114682064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085521104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085520336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085520912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085521296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085521872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085522256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140405085522064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  14

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


CAM export passed the Grad-CAM, score, TFLite map and validation checks.
CAM model: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/deployment/skinx_screening_cam.tflite


,status,keras_cam_gradcam_max_normalized_mae,keras_score_max_absolute_difference,tflite_cam_max_normalized_mae,tflite_score_max_absolute_difference,tflite_cam_minimum_cosine_similarity,tflite_validation_metrics,uncertainty_policy_validation,model_path,config_path
0,PASSED,4.656613e-08,0.0,0.003689,0.004999,0.997823,"{'n': 339, 'tp': 269, 'fn': 17, 'tn': 43, 'fp'...","{'margin': 0.05, 'lower_boundary': 0.324953031...",/content/drive/MyDrive/SkinX/models/runs/noteb...,/content/drive/MyDrive/SkinX/models/runs/noteb...


## 9. Create a synthetic image for Flutter/TFLite parity

A synthetic colour image is used to check preprocessing and compare one TFLite score between Android and Python. It is for engineering tests only and should not be included in the released app. The expected resized float32 tensor and score are saved separately from the mobile model files.

In [10]:
if DEPLOY_READY:
    from PIL import Image
    parity_dir = OUT / 'parity_tests'
    parity_dir.mkdir(exist_ok=True)

    # This synthetic fixture is for engineering tests and contains no patient image.
    hh, ww = 151, 271
    yy, xx = np.indices((hh, ww))
    sample = np.stack([
        (xx * 255 // max(ww-1, 1)).astype(np.uint8),
        (yy * 255 // max(hh-1, 1)).astype(np.uint8),
        ((xx+2*yy) % 256).astype(np.uint8)], axis=-1)
    sample[25:67, 80:128, :] = [18, 160, 220]
    synthetic_file = parity_dir / 'synthetic_source.png'
    Image.fromarray(sample, 'RGB').save(synthetic_file)

    golden_input = load_image(synthetic_file)
    np.save(parity_dir/'expected_input_rgb_float32.npy', golden_input)
    intr, inp, out = interpreter_for(deploy_dir/'skinx_screening.tflite')
    golden_score = run_one(intr, inp, out, golden_input)
    golden = {
        'source_fixture': synthetic_file.name,
        'expected_tensor_file': 'expected_input_rgb_float32.npy',
        'expected_shape': list(golden_input.shape),
        'expected_dtype': str(golden_input.dtype),
        'input_min': float(golden_input.min()),
        'input_max': float(golden_input.max()),
        'expected_target_probability': golden_score,
        'expected_decision': 'Suspicious' if golden_score >= THRESHOLD else 'Less suspicious',
        'test_input_sha256': sha256(synthetic_file),
        'mean_absolute_float_difference_tolerance_for_native_resize': 1.0,
        'note': 'Synthetic only. Android must also compare output score and decision at locked threshold. Resize tolerance is an engineering guide, not a clinical equivalence proof.'
    }
    (parity_dir/'expected_output.json').write_text(json.dumps(golden, indent=2)+'\n', encoding='utf-8')
    print('Saved synthetic Android parity fixture:', parity_dir)
else:
    print('Parity fixture skipped: no deployment-ready model.')

Saved synthetic Android parity fixture: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/parity_tests


/tmp/ipykernel_3632/288410607.py:15: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(sample, 'RGB').save(synthetic_file)
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 10. Run a descriptive check on the PWA images

This check happens after the deployment choice is fixed and is not used to select or tune the converted model. The PWA labels are provisional, some images are small, and there are only five non-target images. The results are a small pilot check, not clinical validation.

In [11]:
PWA_CSV=PWA_ROOT/'metadata.csv'
if DEPLOY_READY and PWA_CSV.exists():
    pwa=pd.read_csv(PWA_CSV)
    for field in ['image_id','filepath','binary_label','split_group_id']:
        if field not in pwa.columns: raise ValueError(f'PWA dataset missing {field}')
    pwa['absolute_path']=pwa['filepath'].map(lambda x: str(PWA_ROOT/str(x)))
    missing=[p for p in pwa['absolute_path'] if not Path(p).is_file()]
    if missing: raise FileNotFoundError(f'{len(missing)} PWA images missing: {missing[0]}')
    if pwa['image_id'].duplicated().any(): raise ValueError('Duplicate PWA ids')
    intr,inp,out=interpreter_for(deploy_dir/'skinx_screening.tflite')
    keras_pwa=[]; tflite_pwa=[]
    for path in pwa['absolute_path']:
        img=load_image(path)
        keras_pwa.append(float(np.asarray(model(img[None,...],training=False)).reshape(-1)[0]))
        tflite_pwa.append(run_one(intr,inp,out,img))
    pwa['keras_probability']=keras_pwa
    pwa['tflite_probability']=tflite_pwa
    pwa['keras_prediction']=(pwa['keras_probability']>=THRESHOLD).astype(int)
    pwa['tflite_prediction']=(pwa['tflite_probability']>=THRESHOLD).astype(int)
    pwa.to_csv(OUT/'pwa_descriptive_predictions.csv',index=False)
    print('PWA check — original Keras:',metrics(pwa['binary_label'],pwa['keras_probability']))
    print('PWA check — chosen TFLite:',metrics(pwa['binary_label'],pwa['tflite_probability']))
    print('Changed PWA decisions:',int((pwa['keras_prediction']!=pwa['tflite_prediction']).sum()))
    print('PWA results are for reporting only, not re-selection.')
else:
    print('PWA descriptive check skipped (no deployment-ready variant or PWA metadata missing).')


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


PWA check — original Keras: {'n': 33, 'tp': 28, 'fn': 0, 'tn': 2, 'fp': 3, 'sensitivity': 1.0, 'specificity': 0.4, 'auroc': 0.9785714285714285}
PWA check — chosen TFLite: {'n': 33, 'tp': 28, 'fn': 0, 'tn': 2, 'fp': 3, 'sensitivity': 1.0, 'specificity': 0.4, 'auroc': 0.9785714285714285}
Changed PWA decisions: 0
PWA results are for reporting only, not re-selection.


## 11. Separate screening decisions from technical errors

The app displays `Suspicious`, `Less suspicious` or `Cannot infer`. The first two are screening labels based on the locked Notebook 04 threshold and the uncertainty band; they are not diagnoses. `Cannot infer` covers technical failures and scores inside the initial 0.05 margin. The margin was checked on conventional validation scores but has not been clinically validated. There is no validated image-quality model or medical blur/lighting cutoff in this notebook. A lesion that changes, bleeds, hurts or does not heal still needs medical assessment, even if the result is `Less suspicious`.

The next cell exports a reusable Python reference for offline testing and optional FastAPI use. It checks JPEG/PNG support, corrupt or truncated files, large payloads and pixel limits. These are technical safeguards, not clinically validated image-quality checks. Phone photos need EXIF orientation corrected before Flutter builds the inference tensor; the Python reference applies this correction when needed. Images, location and results are not automatically uploaded.

In [ ]:
MODULE_CODE = r'''
"""SkinX offline screening reference. NOT the Android inference runtime.
"""
from __future__ import annotations

import hashlib
import io
import json
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps, UnidentifiedImageError
import tensorflow as tf


class InvalidImage(ValueError):
    """Input rejected for a technical reason, not a medical judgment."""


def _verify_and_orient(data: bytes, max_bytes: int, max_pixels: int) -> bytes:
    if not isinstance(data, bytes) or len(data) == 0:
        raise InvalidImage('EMPTY_IMAGE')
    if len(data) > max_bytes:
        raise InvalidImage('IMAGE_TOO_LARGE_BYTES')
    try:
        with Image.open(io.BytesIO(data)) as image:
            if image.format not in ('JPEG', 'PNG'):
                raise InvalidImage('UNSUPPORTED_IMAGE_FORMAT')
            if getattr(image, 'n_frames', 1) != 1:
                raise InvalidImage('ANIMATED_IMAGE_UNSUPPORTED')
            w, h = image.size
            if w <= 0 or h <= 0 or w*h > max_pixels:
                raise InvalidImage('IMAGE_DIMENSIONS_UNSUPPORTED')
            orientation = image.getexif().get(274, 1)
            # Force full decode: Image.open alone does not catch truncated images.
            image.load()
            if orientation != 1:
                # Phone/gallery orientation is normalized BEFORE 224x224 resizing.
                oriented = ImageOps.exif_transpose(image).convert('RGB')
                stream = io.BytesIO()
                oriented.save(stream, format='PNG')
                data = stream.getvalue()
    except InvalidImage:
        raise
    except (UnidentifiedImageError, OSError, SyntaxError, ValueError) as exc:
        raise InvalidImage('INVALID_OR_CORRUPT_IMAGE') from exc
    return data


def preprocess_bytes(data: bytes, config: dict) -> np.ndarray:
    """Exact training resize on already oriented bytes; returns float32 RGB 0..255."""
    raw = _verify_and_orient(
        data,
        max_bytes=int(config['max_upload_bytes']),
        max_pixels=int(config['max_image_pixels']),
    )
    try:
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize_with_pad(tf.cast(image, tf.float32), 224, 224)
        return np.asarray(image.numpy(), dtype=np.float32)
    except Exception as exc:
        raise InvalidImage('UNABLE_TO_PREPROCESS_IMAGE') from exc


class ScreeningEngine:
    """One interpreter per instance. Do not share the same instance concurrently."""
    def __init__(self, model_path, config_path, num_threads=2):
        self.model_path = Path(model_path)
        self.config = json.loads(Path(config_path).read_text(encoding='utf-8'))
        if hashlib.sha256(self.model_path.read_bytes()).hexdigest() != self.config['tflite_sha256']:
            raise ValueError('MODEL_HASH_MISMATCH')
        self.threshold = float(self.config['target_probability_threshold'])
        self.uncertainty_abstention_enabled = bool(self.config.get('uncertainty_abstention_enabled', False))
        self.uncertainty_margin = float(self.config.get('uncertainty_margin', 0.0))
        if not 0 < self.threshold < 1 or self.config['labels'] != {'0':'NON_TARGET','1':'TARGET'}:
            raise ValueError('INVALID_SCREENING_CONFIG')
        if not np.isfinite(self.uncertainty_margin) or self.uncertainty_margin < 0:
            raise ValueError('INVALID_UNCERTAINTY_MARGIN')
        if self.uncertainty_abstention_enabled and (self.uncertainty_margin <= 0 or
                self.uncertainty_margin >= self.threshold or self.uncertainty_margin >= 1-self.threshold):
            raise ValueError('INVALID_UNCERTAINTY_MARGIN')
        self.interpreter = tf.lite.Interpreter(model_path=str(self.model_path), num_threads=num_threads)
        self.interpreter.allocate_tensors()
        inputs, outputs = self.interpreter.get_input_details(), self.interpreter.get_output_details()
        if len(inputs) != 1 or len(outputs) != 1 or tuple(inputs[0]['shape']) != (1,224,224,3):
            raise ValueError('UNEXPECTED_MODEL_INPUT')
        if int(np.prod(outputs[0]['shape'])) != 1:
            raise ValueError('UNEXPECTED_MODEL_OUTPUT')
        self.inp, self.out = inputs[0], outputs[0]

    def predict_tensor(self, image: np.ndarray) -> float:
        if image.shape != (224,224,3) or not np.isfinite(image).all():
            raise InvalidImage('INVALID_TENSOR')
        value = np.expand_dims(image, 0).astype(np.float32)
        if np.issubdtype(self.inp['dtype'], np.integer):
            scale, zero = self.inp['quantization']
            if scale <= 0:
                raise ValueError('INVALID_INPUT_QUANTIZATION')
            limits = np.iinfo(self.inp['dtype'])
            value = np.clip(np.rint(value/scale + zero), limits.min, limits.max).astype(self.inp['dtype'])
        else:
            value = value.astype(self.inp['dtype'])
        self.interpreter.set_tensor(self.inp['index'], value)
        self.interpreter.invoke()
        output = self.interpreter.get_tensor(self.out['index'])
        if np.issubdtype(self.out['dtype'], np.integer):
            scale, zero = self.out['quantization']
            if scale <= 0:
                raise ValueError('INVALID_OUTPUT_QUANTIZATION')
            output = (output.astype(np.float32)-zero)*scale
        score = float(np.asarray(output).reshape(-1)[0])
        if not np.isfinite(score) or score < -0.005 or score > 1.005:
            raise ValueError('INVALID_MODEL_SCORE')
        return float(np.clip(score, 0.0, 1.0))

    def screen_bytes(self, data: bytes) -> dict:
        try:
            image = preprocess_bytes(data, self.config)
            score = self.predict_tensor(image)
        except InvalidImage as exc:
            return {'status':'Cannot infer', 'target_probability':None,
                    'reason':str(exc), 'message':'Please choose another clear JPEG or PNG image.'}
        except Exception:
            return {'status':'Cannot infer', 'target_probability':None,
                    'reason':'INFERENCE_ERROR', 'message':'Could not process this image. Please try again.'}
        if self.uncertainty_abstention_enabled and abs(score - self.threshold) <= self.uncertainty_margin:
            return {'status':'Cannot infer', 'target_probability':score,
                    'decision_threshold':self.threshold,
                    'uncertainty_lower_boundary':max(0.0, self.threshold-self.uncertainty_margin),
                    'uncertainty_upper_boundary':min(1.0, self.threshold+self.uncertainty_margin),
                    'reason':'BORDERLINE_PREDICTION',
                    'message':'The model score is close to its decision threshold. This does not mean the lesion is harmless; please seek professional assessment.'}
        suspicious_boundary = self.threshold + self.uncertainty_margin if self.uncertainty_abstention_enabled else self.threshold
        if score >= suspicious_boundary:
            return {'status':'Suspicious', 'target_probability':score, 'reason':None,
                    'message':'This image shows a suspicious pattern. Please seek clinical assessment.'}
        return {'status':'Less suspicious', 'target_probability':score, 'reason':None,
                'message':'This screen cannot rule out skin cancer. Seek care for changing, painful, bleeding or non-healing lesions.'}

    def screen_file(self, file_path) -> dict:
        try:
            path = Path(file_path)
            if path.stat().st_size > int(self.config['max_upload_bytes']):
                raise InvalidImage('IMAGE_TOO_LARGE_BYTES')
            return self.screen_bytes(path.read_bytes())
        except (OSError, InvalidImage) as exc:
            return {'status':'Cannot infer', 'target_probability':None,
                    'reason':str(exc) if isinstance(exc, InvalidImage) else 'UNREADABLE_IMAGE',
                    'message':'The image could not be opened.'}

'''

if DEPLOY_READY:
    module_path = OUT / 'skinx_inference.py'
    module_path.write_text(MODULE_CODE, encoding='utf-8')
    compile(module_path.read_text(encoding='utf-8'), str(module_path), 'exec')

    # These are technical limits, not clinically validated image-quality or uncertainty rules.
    config['max_upload_bytes'] = 12 * 1024 * 1024
    config['max_image_pixels'] = 20_000_000
    config['allowed_image_formats'] = ['JPEG','PNG']
    config['technical_image_checks'] = 'File validity, format, dimensions, payload size; no clinical image-quality gate.'
    config['exif_orientation'] = 'Normalize when needed before RGB resize; verify Flutter parity separately.'
    config['uncertainty_abstention_enabled'] = True
    config['uncertainty_margin'] = UNCERTAINTY_MARGIN
    config['uncertainty_lower_boundary'] = max(0.0, THRESHOLD - UNCERTAINTY_MARGIN)
    config['uncertainty_upper_boundary'] = min(1.0, THRESHOLD + UNCERTAINTY_MARGIN)
    config['uncertainty_policy_validation'] = selected_uncertainty
    config['uncertainty_policy_validation_scope'] = 'Conventional validation split only; exploratory prototype policy, not clinical validation.'
    config['cannot_infer_reasons'] = ['EMPTY_IMAGE','IMAGE_TOO_LARGE_BYTES','IMAGE_DIMENSIONS_UNSUPPORTED',
                                     'UNSUPPORTED_IMAGE_FORMAT','ANIMATED_IMAGE_UNSUPPORTED',
                                     'INVALID_OR_CORRUPT_IMAGE','UNABLE_TO_PREPROCESS_IMAGE',
                                     'INFERENCE_ERROR']
    config['threshold_origin'] = 'Notebook 04 conventional validation; locked, never updated using PWA/test.'
    config['display_messages'] = {
        'Suspicious': 'Suspicious pattern. Seek clinical assessment.',
        'Less suspicious': 'This does not rule out cancer. Seek assessment for persistent or changing lesions.',
        'Cannot infer': 'Technical failures or a borderline model score can prevent SkinX from assigning a screening category. A borderline score does not mean the lesion is harmless.'
    }
    (deploy_dir/'skinx_model_config.json').write_text(json.dumps(config,indent=2)+'\n',encoding='utf-8')
    if CAM_EXPORT_READY:
        cam_config_path = deploy_dir/'skinx_cam_model_config.json'
        cam_settings = json.loads(cam_config_path.read_text(encoding='utf-8'))
        for field in ['max_upload_bytes', 'max_image_pixels', 'allowed_image_formats',
                      'technical_image_checks', 'exif_orientation', 'uncertainty_abstention_enabled',
                      'cannot_infer_reasons', 'threshold_origin', 'display_messages']:
            if field in config:
                cam_settings[field] = config[field]
        cam_config_path.write_text(json.dumps(cam_settings,indent=2)+'\n',encoding='utf-8')
    print('Saved reusable Python inference module:', module_path)
else:
    print('No model passed the validation comparison; skipping deployment module.')

Saved reusable Python inference module: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/skinx_inference.py


## 12. Check repeatability, invalid inputs and inference consistency

Run repeated predictions on the same preprocessed tensor and confirm that the inference API matches the frozen TFLite model. Also check empty, corrupt, unsupported and oversized inputs. These checks cover the Colab/Python reference only; they do not verify native Android inference or clinical image quality.

In [13]:
if DEPLOY_READY:
    import importlib.util
    spec = importlib.util.spec_from_file_location('skinx_inference', module_path)
    screening_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screening_module)
    engine = screening_module.ScreeningEngine(
        deploy_dir/'skinx_screening.tflite',
        deploy_dir/'skinx_model_config.json',
        num_threads=NUM_THREADS)

    # Use the synthetic fixture, not a patient or evaluation image.
    synthetic_bytes = synthetic_file.read_bytes()
    image_tensor = screening_module.preprocess_bytes(synthetic_bytes, config)
    np.testing.assert_allclose(image_tensor, golden_input, rtol=0, atol=1e-6)
    repeat = np.asarray([engine.predict_tensor(image_tensor) for _ in range(8)],dtype=np.float64)
    drift = float(np.max(np.abs(repeat-repeat[0])))
    if drift > 1e-6:
        raise RuntimeError(f'Non-repeatable local TFLite prediction: max drift={drift}')
    if abs(float(repeat[0])-golden_score) > 1e-6:
        raise RuntimeError('Exported module did not reproduce original TFLite parity score')

    good = engine.screen_bytes(synthetic_bytes)
    # The fixture score (~0.417) is inside the 0.325–0.425 uncertainty band, so the expected status is 'Cannot infer'.
    assert good['status'] in {'Suspicious','Less suspicious', 'Cannot infer'}
    bad_cases = {
        'empty':b'',
        'invalid_bytes':b'not an image',
        'unsupported_format':None,
    }
    from PIL import Image
    import io
    test_webp = io.BytesIO()
    try:
        Image.new('RGB', (25,35),(40,140,90)).save(test_webp,format='WEBP')
        bad_cases['unsupported_format']=test_webp.getvalue()
    except Exception:
        bad_cases.pop('unsupported_format')
    for case, raw in bad_cases.items():
        result = engine.screen_bytes(raw)
        if result['status'] != 'Cannot infer' or result['target_probability'] is not None:
            raise AssertionError(f'{case} unexpectedly returned {result}')
    # This is a payload-size limit, not a medical image-quality check.
    oversized = b'1'*(int(config['max_upload_bytes'])+1)
    assert engine.screen_bytes(oversized)['status'] == 'Cannot infer'

    # Test the reusable bytes API with one saved validation image.
    # This check does not change the original Keras/TFLite model selection.
    sample_path = Path(val.iloc[0]['filepath'])
    prediction = engine.screen_file(sample_path)
    assert prediction['status'] in {'Suspicious','Less suspicious'}
    sample_resized = screening_module.preprocess_bytes(sample_path.read_bytes(),config)
    direct_score = run_one(*interpreter_for(dest), sample_resized)
    if abs(prediction['target_probability'] - direct_score) > 1e-6:
        raise RuntimeError('Application inference does not match the selected interpreter')

    test_results = {
        'repeat_count':len(repeat),
        'max_probability_drift':drift,
        'synthetic_expected_probability':golden_score,
        'synthetic_actual_probability':float(repeat[0]),
        'validation_image_byte_path_consistency':True,
        'invalid_input_cases_checked':sorted(bad_cases)+['oversized_payload'],
        'status_example_synthetic':good['status'],
        'status_example_validation':prediction['status'],
        'limitations':'Python/Colab only. Android inference parity and low-end device benchmarks remain untested.'
    }
    (OUT/'inference_tests.json').write_text(json.dumps(test_results,indent=2)+'\n',encoding='utf-8')
    print('Repeatability + input checks passed:'); display(pd.DataFrame([test_results]))
else:
    print('Inference tests skipped. A valid candidate must pass model selection first.')

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Repeatability + input checks passed:


,repeat_count,max_probability_drift,synthetic_expected_probability,synthetic_actual_probability,validation_image_byte_path_consistency,invalid_input_cases_checked,status_example_synthetic,status_example_validation,limitations
0,8,0.0,0.417479,0.417479,True,"[empty, invalid_bytes, unsupported_format, ove...",Cannot infer,Suspicious,Python/Colab only. Android inference parity an...


## 13. Prepare the optional FastAPI reference and Flutter contract

The Android model is the offline screening component. FastAPI is a separate reference for engineering tests, and Flutter screening must not depend on it. The server example is exported separately from the mobile-only ZIP to keep the app bundle small.

In [14]:
if DEPLOY_READY:
    backend = OUT / 'backend_optional'
    backend.mkdir(exist_ok=True)
    shutil.copy2(module_path, backend/'skinx_inference.py')

    sample_api = """# Optional online diagnostic/testing route; NOT used by offline Flutter screening.
from pathlib import Path
from fastapi import FastAPI, UploadFile, File
from skinx_inference import ScreeningEngine

assets = Path(__file__).resolve().parent / 'assets'
engine = ScreeningEngine(assets/'skinx_screening.tflite',
                         assets/'skinx_model_config.json')
app = FastAPI(title='SkinX optional inference reference')

@app.post('/screen')
async def screen(file: UploadFile = File(...)):
    # Limit upload streaming in production reverse proxy too.
    max_bytes = int(engine.config['max_upload_bytes'])
    raw = await file.read(max_bytes+1)
    return engine.screen_bytes(raw)
"""
    (backend/'fastapi_example.py').write_text(sample_api,encoding='utf-8')
    (backend/'requirements_optional.txt').write_text(
        f'tensorflow=={tf.__version__}\nPillow\nnumpy\nfastapi\nuvicorn\npython-multipart\n',encoding='utf-8')
    (backend/'README.md').write_text("""# Optional Python/FastAPI reference

Screening in the Android app is **OFFLINE** and does not call this API.
For optional server testing only, copy `skinx_screening.tflite` and
`skinx_model_config.json` to `backend_optional/assets/` before launch.
`skinx_inference.py` is the single Python source of truth for input validation,
preprocessing and the decision label; it needs TensorFlow on the server.
For production, add authentication, rate limits, file size limits at the proxy,
secure storage controls, privacy review, and tests before exposing any endpoint.
No photos or patient data are automatically synced by the offline app.
""",encoding='utf-8')

    flutter_contract = """# Flutter Android offline inference contract

- Ship ONLY the two mobile assets in `SkinX_android_assets.zip`.
  Do not bundle TensorFlow Python or the optional backend.
- Handle capture/selection fully on the device; rotate according to EXIF BEFORE resize.
- Validate JPEG/PNG bytes and reject corrupt/unsupported files. The allowed formats
  and resource limits in the JSON are TECHNICAL guards, not medical quality scores.
- Pixel contract: 3-channel RGB, **224×224**, bilinear aspect-ratio resize,
  centered **black** padding. Float pixel range remains **0–255**.
- Read `input_dtype`, `input_quantization`, `output_dtype`, `output_quantization`
  from the saved config to quantize/dequantize. Do not divide by 255.
- Use `target_probability_threshold`, `uncertainty_abstention_enabled`, and
  `uncertainty_margin` exactly as saved. When abstention is enabled, scores within
  the margin around the threshold return Cannot infer; scores above the upper
  boundary return Suspicious and scores below the lower boundary return Less suspicious.
  This is a prototype policy evaluated on conventional validation only, not clinical validation.
- Compare Flutter's preprocessing on `parity_tests/synthetic_source.png` to
  `parity_tests/expected_input_rgb_float32.npy`, then its output to
  `parity_tests/expected_output.json` before accepting Android implementation.
- Test on physical low-memory, older Android devices for cold/hot time, RAM,
  model loading, camera orientation and battery cost. Colab timing is not that test.
- Store history locally and securely if consented; internet is optional, not a dependency.
- Low target probability is not a medical clearance. Even with a Less suspicious
  result, non-healing/changing lesions need a clinician.
- Do NOT show probabilities as a confirmed diagnosis or claim histopathology-level safety.
"""
    (OUT/'Flutter_offline_integration_contract.md').write_text(flutter_contract,encoding='utf-8')
    print('Optional reference + Flutter implementation contract exported:',backend)
else:
    print('Optional backend reference skipped: no accepted TFLite model.')

Optional reference + Flutter implementation contract exported: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/backend_optional


## 14. Save the mobile files and experiment records

The mobile bundle includes the original screening model and config. If the CAM checks pass, it also includes the CAM model and config. Flutter uses the CAM model when both CAM files are present and falls back to the original model otherwise.

Keep the Python source, optional API example, repeatability report, validation comparisons, PWA check, CAM validation report, parity vectors and SHA-256 manifest in the notebook run folder. No patient images are included in the Android assets, and offline inference makes no cloud requests.

In [15]:
summary = {
    'run_tag':RUN_TAG,
    'source_lock_sha256':sha256(LOCK_JSON),
    'source_model_sha256':lock['model_sha256'],
    'train_manifest_sha256':sha256(TRAIN_CSV),
    'validation_manifest_sha256':sha256(VAL_CSV),
    'threshold_unchanged':THRESHOLD,
    'keras_validation_metrics':reference,
    'conversion_report':conversion,
    'deployment_ready':DEPLOY_READY,
    'cam_export_ready':CAM_EXPORT_READY,
    'cam_validation_report':CAM_VALIDATION_REPORT,
    'tested_in_python_on_colab':bool(DEPLOY_READY),
    'android_tested':False,
    'uncertainty_abstention_validated':False,
    'uncertainty_policy_evaluated_on_validation':bool(DEPLOY_READY),
    'uncertainty_policy_margin':UNCERTAINTY_MARGIN,
    'uncertainty_policy_report':str(OUT/'uncertainty_policy_validation.json') if DEPLOY_READY else None,
    'medical_image_quality_classifier_validated':False,
    'no_pwa_or_test_tuning':True,
    'tensorflow_version':tf.__version__,
    'numpy_version':np.__version__,
    'pandas_version':pd.__version__,
    'note':'Model selection based on conventional validation only; no proven clinical safety or on-device benchmarks.'
}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
if DEPLOY_READY:
    mobile_files = ['skinx_screening.tflite', 'skinx_model_config.json']
    if CAM_EXPORT_READY:
        mobile_files += ['skinx_screening_cam.tflite', 'skinx_cam_model_config.json']
    for item in mobile_files:
        if not (deploy_dir/item).exists():
            raise FileNotFoundError(f'Missing mobile asset {item}')
    model_hash = sha256(deploy_dir/'skinx_screening.tflite')
    if model_hash != config['tflite_sha256']:
        raise ValueError('Model hash mismatch before mobile export')
    mobile_archive = OUT / 'SkinX_android_assets.zip'
    from zipfile import ZipFile, ZIP_DEFLATED
    with ZipFile(mobile_archive, 'w', compression=ZIP_DEFLATED) as archive:
        for item in mobile_files:
            archive.write(deploy_dir/item,arcname=item)

    (OUT/'artifact_sha256.json').write_text(json.dumps({
        'skinx_screening.tflite':model_hash,
        'skinx_model_config.json':sha256(deploy_dir/'skinx_model_config.json'),
        'skinx_inference.py':sha256(module_path),
        'SkinX_android_assets.zip':sha256(mobile_archive),
        'synthetic_source.png':sha256(synthetic_file),
        'expected_input_rgb_float32.npy':sha256(parity_dir/'expected_input_rgb_float32.npy'),
    },indent=2)+'\n',encoding='utf-8')
    if CAM_EXPORT_READY:
        artifact_hashes = json.loads((OUT/'artifact_sha256.json').read_text(encoding='utf-8'))
        artifact_hashes['skinx_screening_cam.tflite'] = sha256(deploy_dir/'skinx_screening_cam.tflite')
        artifact_hashes['skinx_cam_model_config.json'] = sha256(deploy_dir/'skinx_cam_model_config.json')
        (OUT/'artifact_sha256.json').write_text(json.dumps(artifact_hashes,indent=2)+'\n',encoding='utf-8')
    print('READY FOR FLUTTER IMPLEMENTATION (NOT clinically cleared).')
    print('Mobile-only bundle:',mobile_archive)
    print('Module for parity/optional FastAPI:',module_path)
    print('Android physical-device performance and Flutter parity are still REQUIRED.')
else:
    print('NOT READY: no conversion preserved original validation TP/TN.')
    print('See conversion_report.json and validation_comparison.csv; keep original locked model.')
print('Notebook 06 output:', OUT)

READY FOR FLUTTER IMPLEMENTATION (NOT clinically cleared).
Mobile-only bundle: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/SkinX_android_assets.zip
Module for parity/optional FastAPI: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2/skinx_inference.py
Android physical-device performance and Flutter parity are still REQUIRED.
Notebook 06 output: /content/drive/MyDrive/SkinX/models/runs/notebook_06/offline_tflite_cam_v2
